# 🏦 Clasificación de Área de Gasto/Ingreso mediante PLN (v2)

Este notebook es una versión actualizada y lista para ejecutar desde cero para la clasificación multiclase de transacciones bancarias. 

## Novedades en v2:
- Compatibilidad total con **scikit-learn 1.4+** (eliminación de `multi_class` en LogisticRegression).
- Celda inicial de instalación de dependencias.
- Mejora en la visualización de la importancia de características.
- Estructura simplificada para ejecución directa.

---

## 1. Instalación de Dependencias

In [1]:
# Descomenta la siguiente línea si necesitas instalar las librerías
# !pip install pandas numpy scikit-learn matplotlib seaborn

## 2. Importaciones y Configuración

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

SEED = 42
print("✅ Entorno configurado correctamente")

AttributeError: module 'numpy' has no attribute '__version__'

## 3. Carga y Preparación de Datos

In [ ]:
# Ruta al dataset (suponiendo ejecución desde el directorio 'playground')
DATA_PATH = os.path.join("..", "data", "raw", "db_orig.csv")

try:
    df_raw = pd.read_csv(DATA_PATH)
    print(f"📊 Dataset cargado: {df_raw.shape[0]} filas")
except FileNotFoundError:
    print("❌ Error: No se encontró db_orig.csv en la ruta esperada. Verifica la ubicación del archivo.")

# Filtramos solo las columnas de interés y eliminamos nulos
df = df_raw[["Description", "Area"]].copy().dropna()
print(f"📋 Dataset limpio: {df.shape[0]} filas")
print(f"🏷️  Clases únicas: {df['Area'].nunique()}")
df.head()

## 4. Análisis Exploratorio (EDA)

In [ ]:
# Distribución de clases
plt.figure(figsize=(10, 6))
area_order = df["Area"].value_counts().index
sns.countplot(data=df, y="Area", order=area_order, palette="viridis")
plt.title("Distribución de Transacciones por Área", fontweight="bold")
plt.show()

## 5. División Train / Test (80/20)

In [ ]:
X = df["Description"]
y = df["Area"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f"🔀 Split completado: Train={len(X_train)}, Test={len(X_test)}")

## 6. Definición de Modelos (Pipelines)

Configuramos 5 modelos competitivos usando técnicas clásicas de PLN.

In [ ]:
def create_pipelines():
    return {
        "Naive Bayes": Pipeline([
            ("vec", CountVectorizer(ngram_range=(1, 2), max_features=5000, strip_accents='unicode')),
            ("clf", MultinomialNB(alpha=0.1))
        ]),
        "LR (TF-IDF)": Pipeline([
            ("vec", TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)),
            ("clf", LogisticRegression(max_iter=1000, random_state=SEED))
        ]),
        "SVM (TF-IDF)": Pipeline([
            ("vec", TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)),
            ("clf", LinearSVC(random_state=SEED))
        ]),
        "Random Forest": Pipeline([
            ("vec", TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
            ("clf", RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))
        ]),
        "HistGradientBoosting": Pipeline([
            ("vec", TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
            ("to_dense", FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
            ("clf", HistGradientBoostingClassifier(max_iter=200, random_state=SEED))
        ])
    }

pipelines = create_pipelines()
print("🤖 Pipelines creados correctamente")

## 7. Entrenamiento y Evaluación

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = []

for name, pipe in pipelines.items():
    print(f"🔄 Entrenando {name}...")
    
    # CV Accuracy
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    
    # Fit final
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    results.append({
        "Modelo": name,
        "CV Acc (media)": scores.mean(),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test F1 (weighted)": f1_score(y_test, y_pred, average='weighted')
    })

df_results = pd.DataFrame(results).sort_values("Test Accuracy", ascending=False)
display(df_results.style.highlight_max(subset=["Test Accuracy"], color='lightgreen'))

## 8. Análisis del Mejor Modelo

In [ ]:
best_model_name = df_results.iloc[0]["Modelo"]
best_pipe = pipelines[best_model_name]
y_pred = best_pipe.predict(X_test)

print(f"🏆 Mejor modelo: {best_model_name}")
print("═" * 60)
print(classification_report(y_test, y_pred))

# Matriz de Confusión
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap="Blues", xticks_rotation=45)
plt.title(f"Matriz de Confusión: {best_model_name}", fontweight="bold")
plt.show()

## 9. Importancia de Palabras (Coeficientes LR)

In [ ]:
lr_pipe = pipelines["LR (TF-IDF)"]
vec = lr_pipe.named_steps["vec"]
clf = lr_pipe.named_steps["clf"]

features = vec.get_feature_names_out()
target_names = clf.classes_

# Mostramos top 10 palabras para las 3 áreas más frecuentes
top_areas = df["Area"].value_counts().head(3).index

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, area in enumerate(top_areas):
    area_idx = list(target_names).index(area)
    coefs = clf.coef_[area_idx]
    top_idx = np.argsort(coefs)[-10:]
    
    axes[i].barh(features[top_idx], coefs[top_idx], color="skyblue")
    axes[i].set_title(f"Palabras clave para: {area}", fontweight="bold")

plt.tight_layout()
plt.show()

## 10. Prueba de Inferencia

Introduce tus propias descripciones para ver qué área predice el modelo.

In [ ]:
test_samples = [
    "Cena en restaurante italiano",
    "Pago de suscripción Netflix",
    "Transferencia de nómina",
    "Compra de billetes de avión para verano",
    "Inversión en criptomonedas",
    "Factura de la luz e internet"
]

preds = best_pipe.predict(test_samples)

for text, p in zip(test_samples, preds):
    print(f"📝 '{text}' → Predicción: {p}")